# swapface sur Colab

Remplacement de visage sur video. ComfyUI et ReActor tournent **ici**, sur le GPU
de la session. Tu y accedes depuis ton navigateur par un tunnel, ou depuis ton
Mac avec `piloter.py`.

Depot : https://github.com/kofekod23/swapface

## Marche a suivre

Execute les sections **1 a 10** dans l'ordre, une fois par session. Ensuite tu
travailles dans l'interface ou depuis ton Mac.

## Trois choses a savoir avant de commencer

**La derniere cellule arrete tout.** Ne la lance qu'a la fin. C'est la cause la
plus frequente d'un tunnel qui « tombe ».

**Un GPU ne rend pas forcement plus rapide.** Sur une source 360p a un visage,
un MacBook M5 va plus vite de bout en bout qu'un A100 : la part modele n'est que
de 16 ms sur 513, le reste est du decodage et de l'encodage sur processeur. Le
GPU distant devient rentable en haute resolution, avec plusieurs visages, ou avec
des restaurations enchainees. Les mesures sont dans le README du depot.

**L'URL du tunnel est publique et sans mot de passe.** Quiconque la connait peut
remplir ta file de taches, envoyer des fichiers et lire tes rendus. Ne la diffuse
pas, et arrete le tunnel quand tu as fini.

## 1. Quelle carte as-tu obtenue

A100 et L4 sont interessantes. Sur T4, l'ecart avec un Mac recent est modeste.

Un TPU ne sert a rien ici : `onnxruntime` n'a pas de provider TPU et ReActor
repose sur `onnxruntime` plus PyTorch CUDA. Si tu n'as aucun GPU, va dans
**Execution**, puis **Modifier le type d'execution**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "AUCUN GPU"
!free -g | head -2
!df -h /content | tail -1

## 2. Configuration et boite a outils

Cette cellule ne lance rien. Elle pose les chemins et trois fonctions
reutilisables : demarrer le serveur, ouvrir le tunnel, dire l'etat des deux.

Elles sont autonomes, donc **tu peux relancer le serveur ou le tunnel a tout
moment** sans avoir a rejouer les cellules precedentes. C'est ce qui manquait a
la version precedente de ce notebook.

In [ ]:
import glob
import os
import re
import site
import subprocess
import time
from pathlib import Path

RACINE = Path('/content')
DEPOT = RACINE / 'swapface'
COMFY = RACINE / 'ComfyUI'
NOEUDS = COMFY / 'custom_nodes'
PORT = 8188

JOURNAL_SERVEUR = RACINE / 'comfyui.log'
JOURNAL_TUNNEL = RACINE / 'cloudflared.log'
CLOUDFLARED = RACINE / 'cloudflared'

URL_DEPOT = 'https://github.com/kofekod23/swapface.git'
URL_CLOUDFLARED = ('https://github.com/cloudflare/cloudflared/releases/latest/'
                   'download/cloudflared-linux-amd64')

serveur = None
tunnel = None
adresse = None


def environnement_cuda():
    """onnxruntime-gpu cherche libcudnn et libcublas dans les chemins systeme,
    alors qu'elles arrivent ici par les paquets pip de PyTorch. Sans ce chemin,
    le provider CUDA peut echouer a l'initialisation en silence."""
    dossiers = sorted({os.path.dirname(f)
                       for repertoire in site.getsitepackages()
                       for f in glob.glob(os.path.join(repertoire, 'nvidia', '*', 'lib', '*.so*'))})
    environnement = dict(os.environ)
    if dossiers:
        ancien = environnement.get('LD_LIBRARY_PATH')
        environnement['LD_LIBRARY_PATH'] = ':'.join(dossiers + ([ancien] if ancien else []))
    return environnement, len(dossiers)


def attendre(journal, motif, processus, secondes, quoi):
    """Attend qu'un motif apparaisse dans un journal, ou explique pourquoi non."""
    for _ in range(secondes):
        texte = journal.read_text(errors='ignore') if journal.exists() else ''
        trouve = re.search(motif, texte)
        if trouve:
            return texte, trouve
        if processus.poll() is not None:
            print(texte[-3000:])
            raise RuntimeError(f'{quoi} s\'est arrete, code {processus.poll()}')
        time.sleep(1)
    print(journal.read_text(errors='ignore')[-3000:])
    raise TimeoutError(f'{quoi} : delai depasse, journal ci-dessus')


def demarrer_comfyui():
    """Demarre ComfyUI, ou ne fait rien s'il tourne deja."""
    global serveur
    if serveur is not None and serveur.poll() is None:
        print('ComfyUI tourne deja')
        return serveur

    environnement, nombre = environnement_cuda()
    print(f'{nombre} dossiers de bibliotheques CUDA transmis')
    serveur = subprocess.Popen(
        ['python', 'main.py', '--port', str(PORT)],
        cwd=str(COMFY), env=environnement,
        stdout=JOURNAL_SERVEUR.open('w'), stderr=subprocess.STDOUT,
    )
    texte, _ = attendre(JOURNAL_SERVEUR, r'Starting server', serveur, 300, 'ComfyUI')
    peripherique = re.search(r'Device:\s*(\S+)', texte)
    print('peripherique :', peripherique.group(1) if peripherique else 'inconnu')
    print('serveur pret sur le port', PORT)
    return serveur


def ouvrir_tunnel():
    """Ouvre un tunnel cloudflared et retourne l'URL publique."""
    global tunnel, adresse
    if tunnel is not None and tunnel.poll() is None:
        print('tunnel deja ouvert :', adresse)
        return adresse

    if not CLOUDFLARED.exists():
        import stat
        import urllib.request
        urllib.request.urlretrieve(URL_CLOUDFLARED, CLOUDFLARED)
        CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR)
        print('cloudflared installe')

    tunnel = subprocess.Popen(
        [str(CLOUDFLARED), 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
        stdout=JOURNAL_TUNNEL.open('w'), stderr=subprocess.STDOUT,
    )
    _, trouve = attendre(JOURNAL_TUNNEL, r'https://[a-z0-9-]+\.trycloudflare\.com',
                         tunnel, 90, 'cloudflared')
    adresse = trouve.group(0)
    print('\nURL publique :', adresse)
    print()
    print('NE CLIQUE PAS sur ce lien : un clic depuis Colab envoie')
    print('Sec-Fetch-Site: cross-site, que ComfyUI refuse par un 403.')
    print('Copie l adresse, ouvre un onglet, colle-la dans la barre d adresse.')
    return adresse


def etat():
    """Dit ou en sont les deux processus."""
    for nom, processus in (('ComfyUI', serveur), ('tunnel', tunnel)):
        if processus is None:
            situation = 'jamais demarre dans ce noyau'
        elif processus.poll() is None:
            situation = 'en marche'
        else:
            situation = f'arrete, code {processus.poll()}'
        print(f'{nom:9s} : {situation}')
    if adresse:
        print(f'{"URL":9s} : {adresse}')


def arreter():
    """Arrete les deux processus."""
    global serveur, tunnel
    for nom, processus in (('tunnel', tunnel), ('ComfyUI', serveur)):
        if processus is None or processus.poll() is not None:
            print(f'{nom} deja arrete')
            continue
        processus.terminate()
        try:
            processus.wait(timeout=30)
        except subprocess.TimeoutExpired:
            processus.kill()
        print(f'{nom} arrete')
    serveur = tunnel = None


print('chemins et fonctions en place')

## 3. Disque persistant, facultatif

Sans Drive, les 1,6 Go de modeles se retelechargent a chaque session, environ
cinq minutes. Avec Drive, une fois suffit.

Mets `PERSISTER` a `False` si tu preferes ne rien laisser sur ton Drive.

In [ ]:
PERSISTER = True

if PERSISTER:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELES_PERSISTANTS = Path('/content/drive/MyDrive/swapface/models')
    MODELES_PERSISTANTS.mkdir(parents=True, exist_ok=True)
else:
    MODELES_PERSISTANTS = None

print('modeles persistants :', MODELES_PERSISTANTS)

## 4. Le depot swapface

On reprend `verif_modeles.py`, `telecharger_modeles.py`, `diag_gpu.py` et
`workflow.json` plutot que de les recopier ici. Le depot est public, aucun jeton
n'est demande.

In [ ]:
if DEPOT.exists():
    subprocess.run(['git', '-C', str(DEPOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', URL_DEPOT, str(DEPOT)], check=True)

print('\nfichiers :', sorted(p.name for p in DEPOT.iterdir() if p.is_file()))

## 5. Installation de ComfyUI, ReActor et VideoHelperSuite

Environ trois minutes. PyTorch est deja present sur Colab et compile pour CUDA,
pip le laissera tranquille.

Depots utilises, verifies :

- ComfyUI : `Comfy-Org/ComfyUI`, l'ancienne adresse `comfyanonymous/ComfyUI` y redirige
- ReActor : `Gourieff/ComfyUI-ReActor`, branche `main`, qui n'a plus besoin
  d'insightface ni d'outils de compilation C++
- VideoHelperSuite : `Kosinkadink/ComfyUI-VideoHelperSuite`

In [ ]:
%%time

def cloner(url, destination, branche=None):
    if destination.exists():
        print('deja la :', destination.name)
        return
    commande = ['git', 'clone', '--depth', '1']
    if branche:
        commande += ['-b', branche]
    commande += [url, str(destination)]
    resultat = subprocess.run(commande, capture_output=True, text=True)
    if resultat.returncode != 0:
        print(resultat.stderr[-2000:])
        raise RuntimeError(f'clone echoue : {url}')
    print('clone :', destination.name)

cloner('https://github.com/Comfy-Org/ComfyUI.git', COMFY)
cloner('https://github.com/Gourieff/ComfyUI-ReActor.git',
       NOEUDS / 'ComfyUI-ReActor', branche='main')
cloner('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
       NOEUDS / 'ComfyUI-VideoHelperSuite')

!pip install -q -r {COMFY}/requirements.txt
!pip install -q -r {NOEUDS}/ComfyUI-VideoHelperSuite/requirements.txt
# install.py de ReActor a besoin de pkg_resources, supprime de setuptools 81,
# ou de son repli importlib_metadata.
!pip install -q importlib_metadata

import torch
print('\ntorch', torch.__version__, '| cuda', torch.version.cuda,
      '| disponible', torch.cuda.is_available())

## 6. Modeles

`install.py` de ReActor telecharge `inswapper_128.onnx` et installe
`onnxruntime-gpu`. Il ne pose ni `buffalo_l` ni `codeformer-v0.1.0.pth`, c'est le
role de `telecharger_modeles.py`.

Si tu as monte Drive, le dossier des modeles y est deplace puis relie par un lien
symbolique, **avant** tout telechargement.

In [ ]:
%%time
import shutil

MODELES = COMFY / 'models'

if MODELES_PERSISTANTS is not None and not MODELES.is_symlink():
    if MODELES.exists():
        for element in MODELES.iterdir():
            cible = MODELES_PERSISTANTS / element.name
            if not cible.exists():
                shutil.move(str(element), str(cible))
        shutil.rmtree(MODELES)
    MODELES.symlink_to(MODELES_PERSISTANTS)
    print('models relie a', MODELES_PERSISTANTS)

subprocess.run(['python', 'install.py'],
               cwd=str(NOEUDS / 'ComfyUI-ReActor'), check=True)

# telecharger_modeles.py attend un dossier ComfyUI a cote de lui.
lien = DEPOT / 'ComfyUI'
if not lien.exists():
    lien.symlink_to(COMFY)
subprocess.run(['python', 'telecharger_modeles.py', '--hyperswap'],
               cwd=str(DEPOT), check=True)

## 7. La bonne roue onnxruntime

**Cette cellule est indispensable et doit passer avant le demarrage du serveur.**

`install.py` de ReActor installe `onnxruntime-gpu` depuis PyPI. Cette roue est
compilee pour **CUDA 13**, alors que Colab tourne en **CUDA 12.8**. L'echec est
silencieux et couteux :

```
providers annonces  : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', ...]
providers effectifs : ['CPUExecutionProvider']
396,3 ms par inference        au lieu de 8,9
```

`get_available_providers()` liste ce que la roue **sait faire**, pas ce qui
**s'initialise**. Le journal du moteur donne la raison :
`Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 13.*`

Le projet onnxruntime publie une variante CUDA 12 sur un index dedie, sous le
meme numero de version. On installe donc la roue par son adresse directe, sinon
la resolution de pip est ambigue entre deux index publiant `1.29.0`.

In [ ]:
import sys

BASE = ('https://aiinfra.pkgs.visualstudio.com/2692857e-05ef-43b4-ba9c-ccf1c22c437c'
        '/_packaging/9387c3aa-d9ad-4513-968c-383f6f7f53b8/pypi/download'
        '/onnxruntime-gpu/1.29/onnxruntime_gpu-1.29.0-{tag}-{tag}-manylinux_2_28_x86_64.whl')

tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
roue = BASE.format(tag=tag)
print('roue CUDA 12 :', roue.rsplit('/', 1)[-1])

# --no-deps : cet index n'heberge pas les dependances, deja installees par ailleurs.
resultat = subprocess.run(
    ['pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps', roue],
    capture_output=True, text=True,
)
if resultat.returncode != 0:
    print(resultat.stdout[-1500:])
    print(resultat.stderr[-1500:])
    raise RuntimeError("installation de la roue CUDA 12 echouee. "
                       "Verifie que ta version de Python a bien une roue publiee.")
print('installe')

## 8. Verification des modeles

Compare les empreintes SHA256 aux valeurs publiees dans le README de ReActor.
Sept lignes attendues. Si l'une manque ou differe, ne va pas plus loin.

In [ ]:
!python {DEPOT}/verif_modeles.py {COMFY}

## 9. Demarrage et tunnel

Le proxy integre de Colab ne convient pas a ComfyUI : il sert la page depuis un
domaine `googleusercontent.com`, les requetes arrivent en
`Sec-Fetch-Site: cross-site` et ComfyUI repond **403**, par protection contre un
site tiers qui viendrait remplir ta file (`server.py`,
`create_origin_only_middleware`).

On passe donc par cloudflared, qui donne une vraie URL publique. Pas de compte,
pas de jeton, contrairement a ngrok qui impose un authtoken.

**Le meme controle interdit d'ouvrir l'URL en cliquant dessus.** Une navigation
declenchee par un clic depuis Colab est elle aussi `cross-site`. Copie l'adresse
et colle-la dans la barre d'adresse d'un nouvel onglet : une adresse saisie a la
main envoie `Sec-Fetch-Site: none`, que ComfyUI accepte.

Contournement si cela t'agace : demarrer ComfyUI avec `--enable-cors-header`.
Le 403 disparait quelle que soit la provenance, mais n'importe quel site que tu
visiterais pourrait alors envoyer des requetes a ton tunnel s'il en connait
l'URL. Le copier-coller ne coute rien, prefere-le.

Limites annoncees par Cloudflare pour ces tunnels rapides : 200 requetes
simultanees, pas de Server-Sent Events, aucune garantie de disponibilite.
ComfyUI passe par WebSocket, donc cela convient.

In [ ]:
demarrer_comfyui()
ouvrir_tunnel()

## 10. Verifier que le GPU travaille vraiment

**Ne saute pas cette etape.** Le panneau de ressources de Colab affiche de la
memoire, pas de l'utilisation : il ne repond pas a la question.

Attendu sur A100 : `CUDAExecutionProvider` sur la quasi-totalite des noeuds, et
`hyperswap_1a_256` autour de 9 ms. Si tu lis 150 ms ou plus, la roue de la
section 7 n'est pas passee.

In [ ]:
!python {DEPOT}/diag_gpu.py --iterations 15

---

# Utiliser l'interface

Ouvre l'URL affichee a la section 9, **directement dans un onglet**.

## Charger le graphe

`workflow.json` est dans le depot, sur ton Mac comme ici. **Glisse-le sur le
canevas** : ComfyUI reconstruit le graphe complet, cable et regle.

Pour le construire a la main, double-clic sur le canevas ouvre la recherche de
noeuds. Il en faut quatre :

| noeud | role |
|---|---|
| `VHS_LoadVideo` | lit la video, en sort les images et l'audio |
| `LoadImage` | charge le visage a poser |
| `ReActorFaceSwap` | fait le remplacement |
| `VHS_VideoCombine` | reassemble la video finale |

Puis quatre liaisons, en tirant du point de sortie vers le point d'entree :

```
VHS_LoadVideo  . IMAGE  ->  ReActorFaceSwap  . input_image
LoadImage      . IMAGE  ->  ReActorFaceSwap  . source_image
ReActorFaceSwap. IMAGE  ->  VHS_VideoCombine . images
VHS_LoadVideo  . AUDIO  ->  VHS_VideoCombine . audio
```

Les couleurs des points guident : violet pour les images, bleu pour l'audio.

**Si ta video n'a pas de piste audio, ne cable pas la liaison AUDIO.** Elle ferait
echouer le rendu entier.

## Envoyer tes fichiers

La cellule ci-dessous depose ce que tu choisis dans le dossier `input` de
ComfyUI. Recharge ensuite la page pour que les listes deroulantes se mettent a
jour.

In [ ]:
from google.colab import files
import shutil

ENTREE = COMFY / 'input'
ENTREE.mkdir(exist_ok=True)

for nom in files.upload():
    shutil.move(nom, ENTREE / nom)

print('\ncontenu de input :')
for element in sorted(ENTREE.iterdir()):
    print(f'  {element.name}  ({element.stat().st_size / 1e6:.1f} Mo)')

## Regler les noeuds

### VHS_LoadVideo

| champ | valeur | pourquoi |
|---|---|---|
| `video` | ta video | |
| `frame_load_cap` | `30` pour essayer, `0` pour tout | ce noeud charge **toutes** les images en memoire d'un coup |
| `skip_first_frames` | `0` | pour attaquer plus loin dans le clip |
| `custom_width` et `custom_height` | `0` et `0` | `0` veut dire ne pas redimensionner. Agrandir une source 360p ne cree aucun detail |
| `select_every_nth` | `1` | `2` pour une pre-visualisation deux fois plus rapide |

`force_size` n'existe plus dans les versions actuelles, malgre ce qu'en disent
beaucoup de tutoriels.

### ReActorFaceSwap

| champ | valeur | pourquoi |
|---|---|---|
| `swap_model` | `hyperswap_1a_256.onnx` | 256 px natif contre 128 pour inswapper |
| `facedetection` | `retinaface_resnet50` | le plus fiable des quatre |
| `face_restore_model` | `none` pour travailler | etape la plus lourde, tu la rallumes a la fin |
| `codeformer_weight` | `0.7` | a `1.0` tu gagnes en nettete et perds en ressemblance |
| `detect_gender_input` et `_source` | `no` | evite de charger un modele de plus |
| `input_faces_index` | `0` | `0,1` si plusieurs visages a remplacer |

### VHS_VideoCombine

| champ | valeur | pourquoi |
|---|---|---|
| `frame_rate` | **la meme que ta source** | une autre valeur decale l'audio |
| `format` | `video/nvenc_h264-mp4` | encodage par la carte NVIDIA. C'est le seul reglage ou le GPU distant apporte ce qu'un Mac n'a pas. `video/h264-mp4` sinon |
| `save_output` | coche | |

Puis **Run**, ou `Ctrl` + `Entree`. Le noeud en cours s'entoure de vert, et
`VHS_VideoCombine` affiche un apercu lisible quand c'est fini.

## Quand ca ne marche pas

| ce que tu vois | ce que ca veut dire |
|---|---|
| **403** en ouvrant l'URL | tu as clique sur le lien, ou tu passes par un cadre ou par le proxy Colab. Copie l'adresse et colle-la dans la barre d'adresse d'un onglet neuf |
| **DNS_PROBE_FINISHED_NXDOMAIN** | ton resolveur DNS ne connait pas encore ce nom. Chaque tunnel cree son enregistrement a la volee, et certaines box mettent le refus en cache. Passe ton Mac sur `1.1.1.1`, ou active le DNS securise dans ton navigateur |
| **error 1033**, ou **530** | le tunnel n'est plus connecte. Va a la section « Relancer » |
| `No faces found` | le detecteur ne voit pas de visage : profil, dos, ou visage trop petit. Change de passage, pas de reglage. `trouver_plan.py` du depot repere les bons passages |
| `VHS failed to extract audio` | ta source n'a pas de piste audio. Enleve la liaison AUDIO |
| le noeud reste rouge | survole-le, le message exact est dans l'infobulle |
| la session ralentit d'un coup | `frame_load_cap` trop grand, tu satures la memoire |
| le GPU a l'air inactif | le panneau Colab montre de la memoire, pas de l'utilisation. Relance la section 10 |

## Piloter depuis ton Mac

Plus sur que l'interface pour un rendu long : si le tunnel coupe, tu perds le
suivi mais **pas le rendu**, ComfyUI continue et le fichier finit dans `output`.

In [ ]:
print('Dans le depot swapface, sur ton Mac :\n')
print(f'  python3 piloter.py --serveur {adresse} \\')
print( '      --video source.mp4 --visage mon_visage.jpg \\')
print( '      --images 0 --largeur 0 --hauteur 0 --sortie rendu.mp4')
print('\nOptions utiles : --modele, --restauration, --poids, --depart')
print('Aide complete  : python3 piloter.py --help')

## Recuperer les rendus

In [ ]:
from google.colab import files

SORTIE = COMFY / 'output'
rendus = sorted(SORTIE.glob('*.mp4'), key=lambda p: p.stat().st_mtime, reverse=True)

if not rendus:
    print('rien dans', SORTIE)
else:
    for element in rendus[:5]:
        print(f'{element.name}  ({element.stat().st_size / 1e6:.1f} Mo)')
    print('\ntelechargement du plus recent :', rendus[0].name)
    files.download(str(rendus[0]))

## Relancer si quelque chose s'est arrete

`etat()` dit ou en sont les deux processus. Les fonctions savent ne rien faire si
tout tourne deja, tu peux relancer la cellule sans risque.

Le tunnel te donnera une **nouvelle URL** : les tunnels rapides ne gardent pas
leur nom.

Si `etat()` affiche « jamais demarre dans ce noyau » alors que tu avais bien
lance le serveur, c'est que le noyau a redemarre. Les fichiers survivent, les
processus non : relance la section 2 puis cette cellule.

In [ ]:
etat()
print()
demarrer_comfyui()
ouvrir_tunnel()

## Arreter, a la fin seulement

> **Ne lance cette cellule que quand tu as fini.** Elle arrete le serveur et le
> tunnel. L'URL sera perdue et il faudra relancer la section precedente.

Colab facture tant que la session tourne. Pense aussi a **Execution**, puis
**Interrompre l'execution**, pour liberer la machine.

In [ ]:
arreter()